### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple scehma types and methods for structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, description, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028D81AD0C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028D81AD16A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    genre:str=Field(description="The Genre of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie)
model_with_structure.invoke("Tell me about the movie Incantation")

Movie(title='Incantation', genre='Horror', year=2022, director='Kevin Ko', rating=6.5)

#### Important property of Pydantic with respect to field validation

If the data type of provided does not match the response of the LLM, it's always going to throw an error. For eg. If the title had any integer, and it's always going to throw an error, that's why str data-type was defined in the title.

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ a movie with details."""
    title:str=Field(..., description="The title of the movie")
    genre:str=Field(..., description="The Genre of the movie")
    year:int=Field(..., description="The year the movie was released")
    director:str=Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
model_with_structure.invoke("Tell me about the movie Godzilla Minus One")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants info about the movie "Godzilla\xa0Minus\xa0One". Need to provide details: director, genre, rating, title, year. Could use the function to get a movie object. Probably the function returns movie details. Let\'s call it.', 'tool_calls': [{'id': 'fc_f496238b-bca7-47fe-9272-2139c8b5fba6', 'function': {'arguments': '{"director":"Takashi Yamazaki","genre":"Science Fiction, Monster, Action","rating":8.5,"title":"Godzilla Minus One","year":2023}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 175, 'total_tokens': 295, 'completion_time': 0.251759003, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.008509242, 'prompt_tokens_details': None, 'queue_time': 0.40689253, 'total_time': 0.260268245}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_c

### Nested Structure

In [5]:
from pydantic import BaseModel, Field

class Actors(BaseModel):
    name:str
    age:int
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actors]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in millions USD as a single float number (e.g., 120.0 for $120 million, NOT 120000000)")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Disney Cars")
response

MovieDetails(title='Cars', year=2006, cast=[Actors(name='Owen Wilson', age=38, role='Lightning McQueen (voice)'), Actors(name='Paul Newman', age=81, role='Doc Hudson (voice)'), Actors(name='Bonnie Hunt', age=45, role='Sally Carrera (voice)'), Actors(name='Larry the Cable Guy', age=43, role='Mater (voice)'), Actors(name='Cheech Marin', age=60, role='Guido (voice)'), Actors(name='John Ratzenberger', age=59, role='Mack (voice)')], genres=['Animation', 'Adventure', 'Comedy'], budget=120000000.0)

### Typed Dict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict (TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of release of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typed_dict = model.with_structured_output(MovieDict)
model_with_typed_dict.invoke("Please provide details for the movie Inception")

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

### Data Classes

A Data Class is a class typically containing mainly data, although there aren't any restrictions. You create it using the @dataclass decorator.

In [18]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### Pydantic method

In [17]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information of a person"""
    name:str = Field(description="The name of the person")
    email:str = Field(description="Email ID of the person")
    phone:str = Field(description="Phone number of the person")

agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto selects provider strategy
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, johndoe@gmail.com, +91 9181716151"}]
})
print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, johndoe@gmail.com, +91 9181716151', additional_kwargs={}, response_metadata={}, id='cc754b30-d5e9-4be7-b59b-12a6dba337ce'), AIMessage(content='{"name":"John Doe","email":"johndoe@gmail.com","phone":"+91 9181716151"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from a string. The response format is ContactInfo JSON schema. Need to output JSON with name, email, phone.\n\nName: John Doe\nEmail: johndoe@gmail.com\nPhone: +91 9181716151\n\nMake compact JSON.\n\nCheck required fields present.\n\nThus output: {"name":"John Doe","email":"johndoe@gmail.com","phone":"+91 9181716151"}'}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 237, 'total_tokens': 365, 'completion_time': 0.272197538, 'completion_tokens_details': {'reasoning_tokens': 91}, 'prompt_time': 0.011524052, 'prompt_tokens_details': None, 'queue_time': 0.346959712, 'total_time': 0.28372159}, 'm

#### TypedDict method

In [20]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)
response = agent.invoke({"messages": [{"role":"user", "content":"Extract details from this contact: John Doe, johndoe@example.com, +91 1234567890"}]
})
response["structured_response"]


{'name': 'John Doe', 'email': 'johndoe@example.com', 'phone': '+91 1234567890'}

#### DataClass method

In [22]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str # The name of the person
    email:str # Email of the person
    phone: str # Phone number of the person

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content":"Extract details from this contact: John Doe, johndoe@example.com, +91 1234567890"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='johndoe@example.com', phone='+91 1234567890')